**GANs: Number Generation**

独热变量 (One-Hot Variables)

In [1]:
import torch

def onehot_encoder(position,depth):
    onehot=torch.zeros((depth,))
    onehot[position]=1
    return onehot

In [2]:
print(onehot_encoder(1,5))

tensor([0., 1., 0., 0., 0.])


In [3]:
def int_to_onehot(number):
    onehot=onehot_encoder(number,100)
    return onehot

In [4]:
onehot75=int_to_onehot(75)
print(onehot75)

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])


In [5]:
def onehot_to_int(onehot):
    num=torch.argmax(onehot)
    return num.item()

In [6]:
print(onehot_to_int(onehot75))

75


In [7]:
def gen_sequence():
    indices = torch.randint(0, 20, (10,))
    values = indices*5
    return values    

In [8]:
sequence=gen_sequence()
print(sequence)

tensor([85, 65, 95, 55, 85, 10, 45, 95, 90, 50])


In [9]:
import numpy as np

def gen_batch():
    sequence=gen_sequence()    #A
    batch=[int_to_onehot(i).numpy() for i in sequence]    #B
    batch=np.array(batch)
    return torch.tensor(batch)
batch=gen_batch()

In [10]:
def data_to_num(data):
    num=torch.argmax(data,dim=-1)    #A
    return num
numbers=data_to_num(batch)    

In [11]:
from torch import nn

device="cuda" if torch.cuda.is_available() else "cpu"

D=nn.Sequential(
    nn.Linear(100,1),
    nn.Sigmoid()).to(device)

In [12]:
G=nn.Sequential(
    nn.Linear(100,100),
    nn.ReLU()).to(device)

In [13]:
loss_fn=nn.BCELoss()
lr=0.0005
optimD=torch.optim.Adam(D.parameters(),lr=lr)
optimG=torch.optim.Adam(G.parameters(),lr=lr)

In [14]:
real_labels=torch.ones((10,1)).to(device)
fake_labels=torch.zeros((10,1)).to(device)

In [15]:
def train_D_G(D,G,loss_fn,optimD,optimG):
    # Generate examples of real data
    true_data=gen_batch().to(device)
    # use 1 as labels since they are real
    preds=D(true_data)
    loss_D1=loss_fn(preds,real_labels.reshape(10,1))
    optimD.zero_grad()
    loss_D1.backward()
    optimD.step()
    # train D on fake data
    noise=torch.randn(10,100).to(device)
    generated_data=G(noise)
    # use 0 as labels since they are fake
    preds=D(generated_data)
    loss_D2=loss_fn(preds,fake_labels.reshape(10,1))
    optimD.zero_grad()
    loss_D2.backward()
    optimD.step()
    
    # train G 
    noise=torch.randn(10,100).to(device)
    generated_data=G(noise)
    # use 1 as labels since G wants to fool D
    preds=D(generated_data)
    loss_G=loss_fn(preds,real_labels.reshape(10,1))
    optimG.zero_grad()
    loss_G.backward()
    optimG.step()
    return generated_data       

In [17]:
class EarlyStop:
    def __init__(self, patience=1000):    #A
        self.patience = patience
        self.steps = 0
        self.min_gdif = float('inf')
    def stop(self, gdif):    #B
        if gdif < self.min_gdif:    #C
            self.min_gdif = gdif
            self.steps = 0
        elif gdif >= self.min_gdif:
            self.steps += 1
        if self.steps >= self.patience:    # 如果模型在1000个轮次内均未发生改进，则停止训练
            return True
        else:
            return False

stopper=EarlyStop(800)

In [18]:
mse=nn.MSELoss()
real_labels=torch.ones((10,1)).to(device)
fake_labels=torch.zeros((10,1)).to(device)
def distance(generated_data):    #B
    nums=data_to_num(generated_data)
    remainders=nums%5
    ten_zeros=torch.zeros((10,1)).to(device)
    mseloss=mse(remainders,ten_zeros)
    return mseloss

for i in range(10000):
    gloss=0
    dloss=0
    generated_data=train_D_G(D,G,loss_fn,optimD,optimG)    #C  
    dis=distance(generated_data)
    if stopper.stop(dis)==True:
        break   
    if i % 50 == 0:
        print(data_to_num(generated_data))    #D

d:\DeepLearning\Anaconda\envs\Generative-Model\lib\site-packages\torch\nn\modules\loss.py:630: UserWarning: Using a target size (torch.Size([10, 1])) that is different to the input size (torch.Size([10])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


tensor([87, 31, 24, 41, 96, 84, 97, 79, 55, 36], device='cuda:0')
tensor([12, 33, 56, 53, 76, 70, 97, 38, 56, 63], device='cuda:0')
tensor([86, 56, 69, 40, 25,  8, 81, 24, 35, 38], device='cuda:0')
tensor([58, 37, 55, 79, 56, 82, 31, 24, 83, 20], device='cuda:0')
tensor([27, 46, 66, 61, 45, 34, 97, 27, 25, 86], device='cuda:0')
tensor([55, 12, 25, 40, 66, 82, 56, 86, 82, 12], device='cuda:0')
tensor([76, 25, 12, 54, 31, 55, 40, 55, 40, 53], device='cuda:0')
tensor([20, 25, 12, 45, 55, 55, 82, 40, 40, 53], device='cuda:0')
tensor([82, 56, 53, 70, 40, 86, 25, 55, 25, 55], device='cuda:0')
tensor([69, 66, 25, 82, 16, 25, 12, 25,  3,  5], device='cuda:0')
tensor([40, 25, 60, 34, 55, 55, 12, 40, 25, 12], device='cuda:0')
tensor([69, 25, 55, 25, 53, 86, 55, 69, 55,  3], device='cuda:0')
tensor([40, 95, 95, 25, 90, 55, 25, 25, 55, 40], device='cuda:0')
tensor([55, 90, 25, 95, 85, 15, 95, 85, 31, 35], device='cuda:0')
tensor([90, 15, 90, 90, 45, 45, 95, 10, 55, 90], device='cuda:0')
tensor([90

In [19]:
# Export to TorchScript
import os
os.makedirs("results", exist_ok=True)
scripted = torch.jit.script(G) 
scripted.save('results/num_gen.pt') 

In [ ]:
new_G=torch.jit.load('results/num_gen.pt',
                     map_location=device)

RecursiveScriptModule(
  original_name=Sequential
  (0): RecursiveScriptModule(original_name=Linear)
  (1): RecursiveScriptModule(original_name=ReLU)
)

In [21]:
# obtain inputs from the latent space
torch.manual_seed(42)
noise=torch.randn((10,100)).to(device)
# feed the input to the generator 
new_data=new_G(noise) 
print(data_to_num(new_data))

tensor([50, 45,  0, 75, 65, 75, 45, 95, 50, 45], device='cuda:0')
